<a href="https://colab.research.google.com/github/himanshi123-lab/hcl/blob/main/himanshi%20singhal(2026b03110271).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Project 15: Movie Theater Box Office & Concession Analysis
Input : cinema_cleaned.csv  (raw dataset)
"""
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

FILE = "cinema_cleaned.csv"


# 1. Load and inspect

print("\n=== 1. LOAD & INSPECT ===")
df = pd.read_csv(FILE)
print("Shape:", df.shape)
print(df.head())
df.info()
print(df.describe(include="all").T)






=== 1. LOAD & INSPECT ===
Shape: (1500, 16)
  Transaction_ID        Movie_Title    Genre Screen_Format   Show_Date Showtime_Slot  Ticket_Quantity  Ticket_Price  Concession_Spend  Occupancy_Pct    Weekday  Is_Weekend Day_Type  Ticket_Revenue  \
0      TXN100294  Quantum Leap Year   Sci-Fi           4DX  12-03-2025       Evening                3         16.84              0.00           84.5  Wednesday           0  Weekday           50.52   
1      TXN100553      Court Jesters   Comedy          IMAX  07-06-2025       Evening                3         15.24              0.00           85.4   Saturday           1  Weekend           45.72   
2      TXN101289       Hero Squad 3   Action   Standard 2D  01-05-2025       Evening                4          9.55             11.78           66.0   Thursday           0  Weekday           38.20   
3      TXN100043       Iron Resolve   Action   Standard 2D  01-06-2025       Evening                3         10.07             10.99           82.7     Su

In [ ]:
from google.colab import files
import os

print("Please click 'Choose Files' below to upload your local 'cinema_cleaned.csv' file:")
uploaded = files.upload()

# Get the name of the uploaded file
for filename in uploaded.keys():
    # Update the CSV_FILE variable dynamically in the global scope
    CSV_FILE = os.path.abspath(filename)
    print(f"\n✔ Success! File '{filename}' uploaded successfully.")
    print(f"The file is saved at: {CSV_FILE}")
    print("You can now run the analysis code cell below!")

Please click 'Choose Files' below to upload your local 'cinema_cleaned.csv' file:


Saving cinema_cleaned.csv to cinema_cleaned (1).csv

✔ Success! File 'cinema_cleaned (1).csv' uploaded successfully.
The file is saved at: /content/cinema_cleaned (1).csv
You can now run the analysis code cell below!


In [ ]:
# ---------------------------------------------------------------
# 2. Missing values and duplicate Transaction_ID
# ---------------------------------------------------------------
print("\n=== 2. MISSING VALUES & DUPLICATES ===")
print("Missing values per column:\n", df.isna().sum())
dup_mask = df.duplicated(subset="Transaction_ID", keep=False)
print("Rows sharing a Transaction_ID:", dup_mask.sum())
print("Fully identical duplicate rows:", df.duplicated().sum())
print(df[dup_mask].sort_values("Transaction_ID").head(10))

# Keep the first record of each Transaction_ID
df = df.drop_duplicates(subset="Transaction_ID", keep="first").copy()
print("Shape after removing duplicate IDs:", df.shape)



=== 2. MISSING VALUES & DUPLICATES ===
Missing values per column:
 Transaction_ID      0
Movie_Title         0
Genre               0
Screen_Format       0
Show_Date           0
Showtime_Slot       0
Ticket_Quantity     0
Ticket_Price        0
Concession_Spend    0
Occupancy_Pct       0
Weekday             0
Is_Weekend          0
Day_Type            0
Ticket_Revenue      0
Combined_Revenue    0
Group_Band          0
dtype: int64
Rows sharing a Transaction_ID: 0
Fully identical duplicate rows: 0
Empty DataFrame
Columns: [Transaction_ID, Movie_Title, Genre, Screen_Format, Show_Date, Showtime_Slot, Ticket_Quantity, Ticket_Price, Concession_Spend, Occupancy_Pct, Weekday, Is_Weekend, Day_Type, Ticket_Revenue, Combined_Revenue, Group_Band]
Index: []
Shape after removing duplicate IDs: (1500, 16)


In [ ]:
# ---------------------------------------------------------------
# 3. Clean missing Concession_Spend (justified strategy)
# ---------------------------------------------------------------
# Strategy: concession spend is right-skewed, so the MEDIAN is more robust
# than the mean. We use the median of the same Genre and Screen_Format
# (falling back to the overall median), because spend differs by these.
print("\n=== 3. CLEAN CONCESSION_SPEND ===")
df["Concession_Spend"] = pd.to_numeric(df["Concession_Spend"], errors="coerce")
n_missing = df["Concession_Spend"].isna().sum()
group_median = df.groupby(["Genre", "Screen_Format"])["Concession_Spend"].transform("median")
df["Concession_Spend"] = df["Concession_Spend"].fillna(group_median)
df["Concession_Spend"] = df["Concession_Spend"].fillna(df["Concession_Spend"].median())
print(f"Filled {n_missing} missing values. Remaining missing:", df["Concession_Spend"].isna().sum())

# Tidy other text/numeric columns
df["Showtime_Slot"] = df["Showtime_Slot"].fillna("Unknown")
df["Occupancy_Pct"] = df["Occupancy_Pct"].fillna(df["Occupancy_Pct"].median())


=== 3. CLEAN CONCESSION_SPEND ===
Filled 0 missing values. Remaining missing: 0


In [ ]:
# ---------------------------------------------------------------
# 4. Show_Date -> datetime, weekday, weekend flag
# ---------------------------------------------------------------
print("\n=== 4. DATE FEATURES ===")
df["Show_Date"] = pd.to_datetime(df["Show_Date"], dayfirst=True, errors="coerce")
print("Unparseable dates:", df["Show_Date"].isna().sum())
df["Weekday"] = df["Show_Date"].dt.day_name()
df["Is_Weekend"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), 1, 0)
df["Day_Type"] = np.where(df["Is_Weekend"] == 1, "Weekend", "Weekday")
print(df[["Show_Date", "Weekday", "Is_Weekend", "Day_Type"]].head())


=== 4. DATE FEATURES ===
Unparseable dates: 0
   Show_Date    Weekday  Is_Weekend Day_Type
0 2025-03-12  Wednesday           0  Weekday
1 2025-06-07   Saturday           1  Weekend
2 2025-05-01   Thursday           0  Weekday
3 2025-06-01     Sunday           1  Weekend
4 2025-05-26     Monday           0  Weekday


In [ ]:
# ---------------------------------------------------------------
# 5. Revenue columns
# ---------------------------------------------------------------
df["Ticket_Revenue"] = df["Ticket_Quantity"] * df["Ticket_Price"]
df["Combined_Revenue"] = df["Ticket_Revenue"] + df["Concession_Spend"]

print("\n=== 5. REVENUE TOTALS ===")
print(f"Total ticket revenue     : {df["Ticket_Revenue"].sum():,.2f}")
print(f"Total concession revenue : {df["Concession_Spend"].sum():,.2f}")
print(f"Total combined revenue   : {df["Combined_Revenue"].sum():,.2f}")



=== 5. REVENUE TOTALS ===
Total ticket revenue     : 55,115.55
Total concession revenue : 18,839.75
Total combined revenue   : 73,955.30


In [ ]:
# ---------------------------------------------------------------
# 6. Top 10 movies by tickets sold and by ticket revenue
# ---------------------------------------------------------------
print("\n=== 6. TOP 10 MOVIES ===")
movie = df.groupby("Movie_Title").agg(
    Tickets_Sold=("Ticket_Quantity", "sum"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
).round(2)
top_by_tickets = movie.sort_values("Tickets_Sold", ascending=False).head(10)
top_by_revenue = movie.sort_values("Ticket_Revenue", ascending=False).head(10)
print("Top 10 by tickets sold:\n", top_by_tickets)
print("\nTop 10 by ticket revenue:\n", top_by_revenue)


=== 6. TOP 10 MOVIES ===
Top 10 by tickets sold:
                    Tickets_Sold  Ticket_Revenue
Movie_Title                                    
Hero Squad 3                563         6202.96
Iron Resolve                497         5674.30
Dino Island                 461         5350.63
Galactic Frontier           392         4649.03
Laugh Track                 342         4034.19
Midnight Whisper            338         4100.18
Magic Kitchen               322         3640.80
The Last Heist              306         3818.20
Quantum Leap Year           301         3494.87
Silent Witness              256         2885.92

Top 10 by ticket revenue:
                    Tickets_Sold  Ticket_Revenue
Movie_Title                                    
Hero Squad 3                563         6202.96
Iron Resolve                497         5674.30
Dino Island                 461         5350.63
Galactic Frontier           392         4649.03
Midnight Whisper            338         4100.18
Laugh Tra

In [ ]:
# ---------------------------------------------------------------
# 7. Average occupancy by screen format
# ---------------------------------------------------------------
print("\n=== 7. OCCUPANCY BY SCREEN FORMAT ===")
screen_summary = df.groupby("Screen_Format").agg(
    Avg_Occupancy=("Occupancy_Pct", "mean"),
    Avg_Ticket_Price=("Ticket_Price", "mean"),
    Avg_Concession=("Concession_Spend", "mean"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
    Transactions=("Transaction_ID", "count"),
).round(2).sort_values("Avg_Occupancy", ascending=False)
print(screen_summary)


=== 7. OCCUPANCY BY SCREEN FORMAT ===
               Avg_Occupancy  Avg_Ticket_Price  Avg_Concession  Ticket_Revenue  Transactions
Screen_Format                                                                               
IMAX                   80.26             14.84           12.65        17995.03           365
4DX                    70.48             16.21           13.10         7299.95           152
Standard 2D            62.88              8.77           12.62        19130.04           679
3D                     55.75             11.75           12.04        10690.53           304


In [ ]:
# ---------------------------------------------------------------
# 8. Weekday vs weekend
# ---------------------------------------------------------------
print("\n=== 8. WEEKDAY vs WEEKEND ===")
day_summary = df.groupby("Day_Type").agg(
    Transactions=("Transaction_ID", "count"),
    Avg_Tickets=("Ticket_Quantity", "mean"),
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Avg_Concession=("Concession_Spend", "mean"),
    Avg_Combined=("Combined_Revenue", "mean"),
).round(2)
print(day_summary)


=== 8. WEEKDAY vs WEEKEND ===
          Transactions  Avg_Tickets  Avg_Ticket_Revenue  Avg_Concession  Avg_Combined
Day_Type                                                                             
Weekday           1099         3.24               37.51           12.63         50.14
Weekend            401         2.97               34.64           12.38         47.02


In [ ]:
# ---------------------------------------------------------------
# 9. Attendance by showtime slot
# ---------------------------------------------------------------
print("\n=== 9. SHOWTIME SLOTS ===")
slot_summary = df.groupby("Showtime_Slot").agg(
    Transactions=("Transaction_ID", "count"),
    Total_Tickets=("Ticket_Quantity", "sum"),
    Avg_Occupancy=("Occupancy_Pct", "mean"),
    Avg_Concession=("Concession_Spend", "mean"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
).round(2).sort_values("Total_Tickets", ascending=False)
print(slot_summary)



=== 9. SHOWTIME SLOTS ===
               Transactions  Total_Tickets  Avg_Occupancy  Avg_Concession  Ticket_Revenue
Showtime_Slot                                                                            
Evening                 818           2581          72.47           12.85        32111.58
Matinee                 380           1211          64.10           11.88        12264.36
Late Night              290            922          52.54           12.73        10268.19
Unknown                  12             38          64.31           10.13          471.42


In [ ]:
# ---------------------------------------------------------------
# 10. Concession attach rate
# ---------------------------------------------------------------
print("\n=== 10. CONCESSION ATTACH RATE ===")
df["Has_Concession"] = df["Concession_Spend"] > 0
attach_rate = df["Has_Concession"].mean() * 100
print(f"Overall attach rate: {attach_rate:.1f}%")
attach_by_day = (df.groupby("Day_Type")["Has_Concession"].mean() * 100).round(1)
attach_by_format = (df.groupby("Screen_Format")["Has_Concession"].mean() * 100).round(1)
attach_by_slot = (df.groupby("Showtime_Slot")["Has_Concession"].mean() * 100).round(1)
print("By day type:\n", attach_by_day)
print("By screen format:\n", attach_by_format)
print("By showtime slot:\n", attach_by_slot)


=== 10. CONCESSION ATTACH RATE ===
Overall attach rate: 65.7%
By day type:
 Day_Type
Weekday    66.4
Weekend    63.8
Name: Has_Concession, dtype: float64
By screen format:
 Screen_Format
3D             62.5
4DX            68.4
IMAX           67.7
Standard 2D    65.5
Name: Has_Concession, dtype: float64
By showtime slot:
 Showtime_Slot
Evening       67.0
Late Night    66.2
Matinee       62.9
Unknown       58.3
Name: Has_Concession, dtype: float64


In [ ]:
# ---------------------------------------------------------------
# 11. Avg ticket and snack spend by genre
# ---------------------------------------------------------------
print("\n=== 11. GENRE COMPARISON ===")
genre_summary = df.groupby("Genre").agg(
    Avg_Ticket_Price=("Ticket_Price", "mean"),
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Avg_Concession=("Concession_Spend", "mean"),
).round(2).sort_values("Avg_Concession", ascending=False)
print(genre_summary)


=== 11. GENRE COMPARISON ===
             Avg_Ticket_Price  Avg_Ticket_Revenue  Avg_Concession
Genre                                                            
Animation               11.53               35.40           15.20
Action                  11.23               37.71           13.66
Sci-Fi                  11.61               38.06           12.69
Comedy                  11.68               35.88           11.47
Thriller                11.80               36.24           11.29
Horror                  12.01               34.16           11.15
Romance                 11.50               40.75           10.71
Documentary             12.49               36.27            8.50


In [ ]:
# ---------------------------------------------------------------
# 12. NumPy stats for total transaction spend (Combined_Revenue)
# ---------------------------------------------------------------
print("\n=== 12. NUMPY STATS: COMBINED REVENUE ===")
spend = df["Combined_Revenue"].to_numpy()
print(f"Mean   : {np.mean(spend):.2f}")
print(f"Median : {np.median(spend):.2f}")
print(f"Std Dev: {np.std(spend, ddof=1):.2f}")


=== 12. NUMPY STATS: COMBINED REVENUE ===
Mean   : 49.30
Median : 40.49
Std Dev: 39.94


In [ ]:
# ---------------------------------------------------------------
# 13. Unusually large group bookings (IQR on Ticket_Quantity)
# ---------------------------------------------------------------
print("\n=== 13. IQR OUTLIERS ON TICKET QUANTITY ===")
q1, q3 = np.percentile(df["Ticket_Quantity"], [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, bounds=({lower}, {upper})")
df["Large_Booking_Outlier"] = df["Ticket_Quantity"] > upper
outliers = df[df["Large_Booking_Outlier"]]
print(f"Outlier bookings: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% of transactions)")
print(outliers[["Transaction_ID", "Movie_Title", "Screen_Format", "Ticket_Quantity", "Combined_Revenue"]]
      .sort_values("Ticket_Quantity", ascending=False).head(10))


=== 13. IQR OUTLIERS ON TICKET QUANTITY ===
Q1=2.0, Q3=4.0, IQR=2.0, bounds=(-1.0, 7.0)
Outlier bookings: 61 (4.1% of transactions)
     Transaction_ID        Movie_Title Screen_Format  Ticket_Quantity  Combined_Revenue
190       TXN100147     The Last Heist          IMAX               29            465.16
970       TXN100824  Quantum Leap Year   Standard 2D               28            297.97
939       TXN101264       Hero Squad 3   Standard 2D               27            294.50
1014      TXN100311        Dino Island   Standard 2D               27            234.36
188       TXN100715        Laugh Track   Standard 2D               26            241.71
995       TXN100371       Iron Resolve   Standard 2D               26            229.96
638       TXN101326    Hearts in Paris            3D               26            310.22
1364      TXN101284      Court Jesters   Standard 2D               25            226.37
796       TXN100641       Summer Fling   Standard 2D               24      

In [ ]:
# ---------------------------------------------------------------
# 14. Ticket-quantity bands with pd.cut()
# ---------------------------------------------------------------
print("\n=== 14. GROUP BANDS ===")
bins = [0, 1, 2, 5, np.inf]
labels = ["Solo (1)", "Couple (2)", "Small Group (3-5)", "Large Group (6+)"]
df["Group_Band"] = pd.cut(df["Ticket_Quantity"], bins=bins, labels=labels)
band_summary = df.groupby("Group_Band", observed=True).agg(
    Transactions=("Transaction_ID", "count"),
    Avg_Concession=("Concession_Spend", "mean"),
    Avg_Combined=("Combined_Revenue", "mean"),
    Attach_Rate_Pct=("Has_Concession", lambda s: s.mean() * 100),
).round(2)
print(band_summary)


=== 14. GROUP BANDS ===
                   Transactions  Avg_Concession  Avg_Combined  Attach_Rate_Pct
Group_Band                                                                    
Solo (1)                    234            6.95         18.45            53.85
Couple (2)                  572           10.12         33.20            62.24
Small Group (3-5)           574           15.67         60.34            72.30
Large Group (6+)            120           20.25        133.43            74.17


In [ ]:
# ---------------------------------------------------------------
# 15. Recommendations (built from the results above)
# ---------------------------------------------------------------
print("\n=== 15. RECOMMENDATIONS ===")
best_fmt = screen_summary["Avg_Occupancy"].idxmax()
worst_fmt = screen_summary["Avg_Occupancy"].idxmin()
slots = slot_summary.drop(index="Unknown", errors="ignore")
best_slot = slots["Total_Tickets"].idxmax()
weak_slot = slots["Total_Tickets"].idxmin()
top_movie = top_by_revenue.index[0]
top_genre_snack = genre_summary["Avg_Concession"].idxmax()
best_band = band_summary["Avg_Concession"].idxmax()
wk_snack = day_summary["Avg_Concession"]

recs = [
    f"1. Screen allocation: {best_fmt} has the highest average occupancy "
    f"({screen_summary.loc[best_fmt, 'Avg_Occupancy']}%). Give it more screenings of top titles such as "
    f"'{top_movie}', and review or reduce {worst_fmt} slots "
    f"({screen_summary.loc[worst_fmt, 'Avg_Occupancy']}%).",
    f"2. Showtime scheduling: {best_slot} sells the most tickets while {weak_slot} sells the fewest. "
    f"Put blockbusters in {best_slot} and use {weak_slot} for discounted or niche screenings.",
    f"3. Weekend vs weekday: average concession spend is {wk_snack.get('Weekend', float('nan'))} on weekends and "
    f"{wk_snack.get('Weekday', float('nan'))} on weekdays. Promote combo offers on the lower-spending day type "
    f"({'Weekend' if wk_snack.get('Weekend', 0) < wk_snack.get('Weekday', 0) else 'Weekday'}) to lift snack revenue.",
    f"4. Combo bundling: '{best_band}' bookings spend the most on concessions, and {top_genre_snack} films "
    f"have the highest snack spend per transaction. Offer group combos and genre-themed bundles.",
    f"5. Attach rate: only {attach_rate:.1f}% of transactions include concessions. Add ticket-plus-snack "
    f"bundles at online checkout. Large bookings ({len(outliers)} outliers) should get pre-order party packs.",
]
for r in recs:
    print(r)

# ---------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------
df.to_csv("cinema_analysis_output.csv", index=False)
with pd.ExcelWriter("cinema_summary_tables.xlsx") as xw:
    top_by_tickets.to_excel(xw, sheet_name="Top10_Tickets")
    top_by_revenue.to_excel(xw, sheet_name="Top10_Revenue")
    screen_summary.to_excel(xw, sheet_name="Screen_Format")
    day_summary.to_excel(xw, sheet_name="Weekday_vs_Weekend")
    slot_summary.to_excel(xw, sheet_name="Showtime_Slots")
    genre_summary.to_excel(xw, sheet_name="Genre")
    band_summary.to_excel(xw, sheet_name="Group_Bands")
print("\nSaved: cinema_analysis_output.csv, cinema_summary_tables.xlsx")


=== 15. RECOMMENDATIONS ===
1. Screen allocation: IMAX has the highest average occupancy (80.26%). Give it more screenings of top titles such as 'Hero Squad 3', and review or reduce 3D slots (55.75%).
2. Showtime scheduling: Evening sells the most tickets while Late Night sells the fewest. Put blockbusters in Evening and use Late Night for discounted or niche screenings.
3. Weekend vs weekday: average concession spend is 12.38 on weekends and 12.63 on weekdays. Promote combo offers on the lower-spending day type (Weekend) to lift snack revenue.
4. Combo bundling: 'Large Group (6+)' bookings spend the most on concessions, and Animation films have the highest snack spend per transaction. Offer group combos and genre-themed bundles.
5. Attach rate: only 65.7% of transactions include concessions. Add ticket-plus-snack bundles at online checkout. Large bookings (61 outliers) should get pre-order party packs.

Saved: cinema_analysis_output.csv, cinema_summary_tables.xlsx
